# Coordinator v3.3 — Final Score Fusion Ensemble

**Key fixes:**
- IF NAB bundle has NO `entities` wrapper — raw dict detected and handled
- `test_slice` extraction with tuple/list handling
- Entity keys: handles `scores_full`/`scoresfull`, `y_full`/`yfull`, `pred_full`/`predfull`
- Extensive per-model debugging for NAB loading
- 4 NAB models (Conv-AE, ECOD, IF, MP) fused instead of 2


In [1]:
# Cell 0 — Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, glob, json, warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve)
warnings.filterwarnings('ignore')
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
RUNROOT = os.path.join(DRIVEROOT, RUNID)
OUTPUTDIR = os.path.join(RUNROOT, 'coordinator_v3', 'predictions')
os.makedirs(OUTPUTDIR, exist_ok=True)
print('RUNROOT:', RUNROOT)
print('OUTPUT :', OUTPUTDIR)


Mounted at /content/drive
RUNROOT: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001
OUTPUT : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/coordinator_v3/predictions


In [2]:
# Cell 1 — Discovery
MODEL_REGISTRY = {
    'xgb_ae_hybrid': {'tags':['xgb_ae_hybrid'],'cc':['*xgb*ae*credit*.joblib'],'nab':[]},
    'lgbm_classifier': {'tags':['lgbm_classifier','memtostrad','memto_strad'],'cc':['*lgbm*credit*.joblib','*memto*credit*.joblib'],'nab':[]},
    'conv_ae': {'tags':['conv_ae','lstm_ae'],'cc':['*conv_ae*credit*.joblib','*lstm*credit*.joblib'],'nab':['*conv_ae*nab*.joblib','*lstm*nab*.joblib']},
    'ecod': {'tags':['ecod'],'cc':['*ecod*credit*.joblib'],'nab':['*ecod*nab*.joblib']},
    'isolation_forest': {'tags':['iforeststrict','iforest_strict'],'cc':['*if*credit*.joblib','*iforest*credit*.joblib'],'nab':['*if*nab*.joblib','*iforest*nab*.joblib']},
    'matrix_profile': {'tags':['matrix_profile','matrixprofile'],'cc':['*mpcreditcard*.joblib','*matrix*credit*.joblib'],'nab':['*mpnab*.joblib','*matrix*nab*.joblib']},
}
def discover():
    found={}
    for mk,info in MODEL_REGISTRY.items():
        for tag in info['tags']:
            pd_=os.path.join(RUNROOT,tag,'predictions')
            if not os.path.isdir(pd_): continue
            for dtype,pats in [('creditcard',info['cc']),('nab',info['nab'])]:
                for pat in pats:
                    for h in glob.glob(os.path.join(pd_,pat)):
                        key=(mk,dtype)
                        if key not in found or os.path.getsize(h)>os.path.getsize(found[key]):
                            found[key]=h
    return found
discovered=discover()
print(f'Discovered {len(discovered)} bundles:')
for (m,d),p in sorted(discovered.items()):
    print(f'  {m:20s}|{d:12s}|{os.path.getsize(p)/1024:8.1f}KB|{os.path.basename(p)}')
cc_models=[m for (m,d) in discovered if d=='creditcard']
nab_models=[m for (m,d) in discovered if d=='nab']
print(f'\nCC: {cc_models}\nNAB: {nab_models}')


Discovered 10 bundles:
  conv_ae             |creditcard  |  1224.5KB|conv_ae_creditcard_strict.joblib
  conv_ae             |nab         |  5018.8KB|lstmae_nab_strict.joblib
  ecod                |creditcard  |  1224.5KB|ecod_creditcard_strict.joblib
  ecod                |nab         |  5018.8KB|ecod_nab_strict.joblib
  isolation_forest    |creditcard  |  1224.5KB|ifcreditcardstrictpointwise.joblib
  isolation_forest    |nab         |  5018.8KB|ifnabstrictpointwise.joblib
  lgbm_classifier     |creditcard  |  1224.8KB|memtostrad_creditcard.joblib
  matrix_profile      |creditcard  |  1224.5KB|mpcreditcardstrictpointwise.joblib
  matrix_profile      |nab         |  5018.8KB|mpnabstrictpointwise.joblib
  xgb_ae_hybrid       |creditcard  |  1225.0KB|xgb_ae_hybrid_creditcard_strict.joblib

CC: ['xgb_ae_hybrid', 'lgbm_classifier', 'conv_ae', 'ecod', 'isolation_forest', 'matrix_profile']
NAB: ['conv_ae', 'ecod', 'isolation_forest', 'matrix_profile']


In [3]:
# DEBUG — Discovery verification
print('=== DISCOVERY VERIFICATION ===')
print(f'Total bundles discovered: {len(discovered)}')
print(f'CC models ({len(cc_models)}): {cc_models}')
print(f'NAB models ({len(nab_models)}): {nab_models}')
print()
# Verify matrix_profile CC is found
if 'matrix_profile' in cc_models:
    print('✓ matrix_profile found in CC models')
else:
    print('✗ WARNING: matrix_profile NOT in CC models — check discovery patterns!')
# Verify expected count
if len(cc_models) >= 6:
    print(f'✓ Expected >=6 CC models, found {len(cc_models)}')
else:
    print(f'✗ WARNING: Expected >=6 CC models, found {len(cc_models)}')
if len(nab_models) >= 4:
    print(f'✓ Expected >=4 NAB models, found {len(nab_models)}')
else:
    print(f'⚠ Expected >=4 NAB models, found {len(nab_models)}')


=== DISCOVERY VERIFICATION ===
Total bundles discovered: 10
CC models (6): ['xgb_ae_hybrid', 'lgbm_classifier', 'conv_ae', 'ecod', 'isolation_forest', 'matrix_profile']
NAB models (4): ['conv_ae', 'ecod', 'isolation_forest', 'matrix_profile']

✓ matrix_profile found in CC models
✓ Expected >=6 CC models, found 6
✓ Expected >=4 NAB models, found 4


In [4]:
# Cell 2 — Flexible key getter
def gv(d, *keys):
    """Get value from dict, trying multiple key variants."""
    for k in keys:
        if k in d: return d[k]
    # fuzzy: strip underscores + lowercase
    for k in keys:
        kn=k.lower().replace('_','')
        for ek,ev in d.items():
            if ek.lower().replace('_','')==kn: return ev
    return None

def load_cc(bundle):
    ents=bundle.get('entities',{})
    ent=ents.get('creditcard') or ents.get('creditcard_test')
    if ent is None and len(ents)==1: ent=list(ents.values())[0]
    if not ent: raise ValueError(f'No CC entity. Keys={list(ents.keys())}')
    s=gv(ent,'scoresfull','scores_full','scores')
    y=gv(ent,'yfull','y_full','ytrue')
    p=gv(ent,'predfull','pred_full','preds')
    r=gv(ent,'originalrowid','original_row_id')
    t=gv(ent,'threshold')
    return {'scores':np.asarray(s,dtype=np.float64),'y_true':np.asarray(y,dtype=np.int8),
            'preds':np.asarray(p,dtype=np.int8) if p is not None else None,
            'originalrowid':np.asarray(r,dtype=np.int64) if r is not None else None,
            'threshold':float(t) if t is not None else None}

def load_nab(raw_bundle, model_key):
    """Load NAB entities from bundle. Handles:
    - IF: raw dict (no 'entities' wrapper)
    - MP/ECOD/Conv-AE: {'entities': {...}} wrapper
    - scores_full with test_slice (ECOD, IF, MP) vs scoresfull test-only (Conv-AE)
    """
    # Step 1: Get the entities dict
    if 'entities' in raw_bundle:
        entities = raw_bundle['entities']
    else:
        # IF stores raw dict: each key is entity_name, value is entity dict
        # Detect: if top-level keys look like entity dicts (have scores_full/scoresfull)
        first_val = next(iter(raw_bundle.values()), None) if raw_bundle else None
        if isinstance(first_val, dict) and (gv(first_val,'scores_full','scoresfull','scores') is not None):
            entities = raw_bundle
            print(f'    [{model_key}] Detected raw dict format (no entities wrapper)')
        else:
            print(f'    [{model_key}] Unknown bundle format. Top keys: {list(raw_bundle.keys())[:5]}')
            return {}

    result = {}
    n_full_series = 0
    n_test_only = 0
    n_skipped = 0

    for eid, ent in entities.items():
        if not isinstance(ent, dict):
            n_skipped += 1
            continue

        sf = gv(ent,'scoresfull','scores_full','scores')
        yf = gv(ent,'yfull','y_full','ytrue','y_true')
        pf = gv(ent,'predfull','pred_full','preds')
        ts = gv(ent,'test_slice','testslice')

        if sf is None or yf is None:
            n_skipped += 1
            continue

        sf = np.asarray(sf, dtype=np.float64)
        yf = np.asarray(yf, dtype=np.int8)
        pf = np.asarray(pf, dtype=np.int8) if pf is not None else None

        # If test_slice exists, extract test portion
        if ts is not None:
            try:
                s_start, s_end = int(ts[0]), int(ts[1])
                if s_end > s_start and s_end <= len(sf):
                    sf = sf[s_start:s_end]
                    yf = yf[s_start:s_end]
                    pf = pf[s_start:s_end] if pf is not None else None
                    n_full_series += 1
                else:
                    n_test_only += 1  # slice looks wrong, use as-is
            except (TypeError, IndexError, ValueError):
                n_test_only += 1  # no valid slice, use as-is
        else:
            n_test_only += 1

        if len(sf) > 0 and len(yf) > 0 and len(sf) == len(yf):
            result[eid] = {'scores': sf, 'y_true': yf, 'preds': pf}

    print(f'    [{model_key}] Loaded {len(result)} entities '
          f'(sliced={n_full_series}, as-is={n_test_only}, skipped={n_skipped})')
    return result

print('Loaders defined.')


Loaders defined.


In [5]:
# Cell 3 — Load all bundles with debugging
print('=== LOADING CC BUNDLES ===')
cc_data = {}
for mk in cc_models:
    try:
        d = load_cc(joblib.load(discovered[(mk,'creditcard')]))
        cc_data[mk] = d
        print(f'  CC {mk}: {len(d["scores"]):,} samples, {int(d["y_true"].sum())} frauds')
    except Exception as e:
        print(f'  ERROR CC {mk}: {e}')

print('\n=== LOADING NAB BUNDLES ===')
nab_data = {}
for mk in nab_models:
    path = discovered[(mk,'nab')]
    print(f'\n  Loading {mk} from {os.path.basename(path)}...')
    try:
        raw = joblib.load(path)
        print(f'    Raw bundle type: {type(raw).__name__}, top keys: {list(raw.keys())[:8]}')

        # Debug: check first entity
        test_ents = raw.get('entities', raw)  # try both wrapper and raw
        if isinstance(test_ents, dict):
            first_key = next(iter(test_ents.keys()), None)
            if first_key:
                first_val = test_ents[first_key]
                if isinstance(first_val, dict):
                    print(f'    First entity key: {first_key}')
                    print(f'    First entity dict keys: {list(first_val.keys())}')

        ents = load_nab(raw, mk)
        if ents:
            nab_data[mk] = ents
            tp = sum(len(e['scores']) for e in ents.values())
            na = sum(int(e['y_true'].sum()) for e in ents.values())
            print(f'    TOTAL: {len(ents)} entities, {tp:,} pts, {na} anomalies')
        else:
            print(f'    FAILED: 0 entities loaded')
    except Exception as e:
        import traceback
        print(f'    ERROR: {e}')
        traceback.print_exc()

print(f'\n=== SUMMARY ===')
print(f'CC loaded : {sorted(cc_data.keys())}')
print(f'NAB loaded: {sorted(nab_data.keys())}')
for mk in nab_data:
    print(f'  {mk}: {len(nab_data[mk])} entities')


=== LOADING CC BUNDLES ===
  CC xgb_ae_hybrid: 56,962 samples, 98 frauds
  CC lgbm_classifier: 56,962 samples, 98 frauds
  CC conv_ae: 56,962 samples, 98 frauds
  CC ecod: 56,962 samples, 98 frauds
  CC isolation_forest: 56,962 samples, 98 frauds
  CC matrix_profile: 56,962 samples, 98 frauds

=== LOADING NAB BUNDLES ===

  Loading conv_ae from lstmae_nab_strict.joblib...
    Raw bundle type: dict, top keys: ['dataset', 'model', 'protocol', 'entities']
    First entity key: art_daily_no_noise.csv
    First entity dict keys: ['entity_id', 'nab_key', 'scores_full', 'y_full', 'pred_full', 'row_id', 'val_slice', 'test_slice', 'threshold', 'val_f1']
    [conv_ae] Loaded 58 entities (sliced=58, as-is=0, skipped=0)
    TOTAL: 58 entities, 109,655 pts, 11814 anomalies

  Loading ecod from ecod_nab_strict.joblib...
    Raw bundle type: dict, top keys: ['dataset', 'model', 'protocol', 'entities']
    First entity key: art_daily_no_noise.csv
    First entity dict keys: ['entity_id', 'nab_key', 's

In [7]:
# Cell 4 — Align CC
ref_mk=list(cc_data.keys())[0]; ref_y=cc_data[ref_mk]['y_true']; ref_rid=cc_data[ref_mk].get('originalrowid')
print(f'Ref: {ref_mk} ({len(ref_y):,} samples, {int(ref_y.sum())} frauds)')
for mk,d in cc_data.items():
    if mk==ref_mk: continue
    if np.array_equal(d['y_true'],ref_y): print(f'  {mk}: OK')
    else:
        rid=d.get('originalrowid')
        if rid is not None and ref_rid is not None and np.array_equal(np.sort(rid),np.sort(ref_rid)):
            inv=np.empty_like(np.argsort(ref_rid)); inv[np.argsort(ref_rid)]=np.arange(len(ref_rid))
            m=inv[np.argsort(rid)]
            for k in ['scores','y_true','preds','originalrowid']:
                if d[k] is not None: d[k]=d[k][m]
            print(f'  {mk}: reordered')
        else: print(f'  WARNING {mk}: mismatch!')
print('Aligned.')


Ref: xgb_ae_hybrid (56,962 samples, 98 frauds)
  lgbm_classifier: OK
  conv_ae: OK
  ecod: OK
  isolation_forest: OK
  matrix_profile: OK
Aligned.


In [8]:
# Cell 5 — Individual CC Metrics
print('='*80)
print('INDIVIDUAL MODEL METRICS — CREDIT CARD')
print('='*80)
cc_metrics={}
for mk in sorted(cc_data.keys()):
    d=cc_data[mk]
    p=d['preds'] if d['preds'] is not None else (d['scores']>=(d['threshold'] or np.percentile(d['scores'],99))).astype(np.int8)
    m={'precision':precision_score(d['y_true'],p,zero_division=0),
       'recall':recall_score(d['y_true'],p,zero_division=0),
       'f1':f1_score(d['y_true'],p,zero_division=0),
       'rocauc':roc_auc_score(d['y_true'],d['scores']) if len(np.unique(d['y_true']))==2 else float('nan'),
       'prauc':average_precision_score(d['y_true'],d['scores']) if len(np.unique(d['y_true']))==2 else float('nan')}
    cc_metrics[mk]=m
    print(f"  {mk:20s} P={m['precision']:.4f} R={m['recall']:.4f} F1={m['f1']:.4f} ROC={m['rocauc']:.4f} PRAUC={m['prauc']:.4f}")
display(pd.DataFrame([{'model':k,**v} for k,v in cc_metrics.items()]).sort_values('f1',ascending=False))


INDIVIDUAL MODEL METRICS — CREDIT CARD
  conv_ae              P=0.7216 R=0.7143 F1=0.7179 ROC=0.9575 PRAUC=0.6874
  ecod                 P=0.3421 R=0.5306 F1=0.4160 ROC=0.9616 PRAUC=0.3536
  isolation_forest     P=0.4640 R=0.5918 F1=0.5202 ROC=0.9596 PRAUC=0.5065
  lgbm_classifier      P=0.9302 R=0.8163 F1=0.8696 ROC=0.9372 PRAUC=0.8480
  matrix_profile       P=0.8211 R=0.7959 F1=0.8083 ROC=0.9646 PRAUC=0.6957
  xgb_ae_hybrid        P=0.9101 R=0.8265 F1=0.8663 ROC=0.9809 PRAUC=0.8731


,model,precision,recall,f1,rocauc,prauc
3,lgbm_classifier,0.930233,0.816327,0.869565,0.937231,0.848038
5,xgb_ae_hybrid,0.910112,0.826531,0.866310,0.980868,0.873092
4,matrix_profile,0.821053,0.795918,0.808290,0.964591,0.695749
0,conv_ae,0.721649,0.714286,0.717949,0.957455,0.687445
2,isolation_forest,0.464000,0.591837,0.520179,0.959610,0.506539
1,ecod,0.342105,0.530612,0.416000,0.961591,0.353613


In [9]:
# Cell 6 — CC Fusion (improved: top-K adaptive, multi-strategy)
model_keys_cc=sorted(cc_data.keys()); y_cc=ref_y.copy()
raw_scores=np.column_stack([cc_data[m]['scores'] for m in model_keys_cc])

def lr_stack(X,y,names,C=1.0,label=''):
    sc=MinMaxScaler(); Xs=sc.fit_transform(X)
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
    probs=np.zeros(len(y),dtype=np.float64)
    for tr,te in cv.split(Xs,y):
        lr=LogisticRegression(class_weight='balanced',max_iter=2000,random_state=RANDOM_STATE,C=C)
        lr.fit(Xs[tr],y[tr]); probs[te]=lr.predict_proba(Xs[te])[:,1]
    pr,rc,th=precision_recall_curve(y,probs)
    f1s=2*pr[:-1]*rc[:-1]/(pr[:-1]+rc[:-1]+1e-12)
    bi=int(np.nanargmax(f1s)); bt=float(th[bi])
    pds=(probs>=bt).astype(np.int8)
    m={'precision':precision_score(y,pds,zero_division=0),'recall':recall_score(y,pds,zero_division=0),
       'f1':f1_score(y,pds,zero_division=0),'rocauc':roc_auc_score(y,probs),'prauc':average_precision_score(y,probs)}
    print(f'  {label}: P={m["precision"]:.4f} R={m["recall"]:.4f} F1={m["f1"]:.4f} ROC={m["rocauc"]:.4f}')
    return m,probs,pds,bt

def wavg(X,y,names,f1s_list,label=''):
    sc=MinMaxScaler(); Xs=sc.fit_transform(X)
    w=np.array([f**2 for f in f1s_list]); w=w/w.sum()
    ws=(Xs*w[None,:]).sum(axis=1)
    pr,rc,th=precision_recall_curve(y,ws)
    f1s=2*pr[:-1]*rc[:-1]/(pr[:-1]+rc[:-1]+1e-12)
    bi=int(np.nanargmax(f1s)); bt=float(th[bi])
    pds=(ws>=bt).astype(np.int8)
    m={'precision':precision_score(y,pds,zero_division=0),'recall':recall_score(y,pds,zero_division=0),
       'f1':f1_score(y,pds,zero_division=0),'rocauc':roc_auc_score(y,ws),'prauc':average_precision_score(y,ws)}
    print(f'  {label}: P={m["precision"]:.4f} R={m["recall"]:.4f} F1={m["f1"]:.4f} ROC={m["rocauc"]:.4f} W={dict(zip(names,w.round(3)))}')
    return m,ws,pds,bt

print('='*80+'\nCC FUSION EXPERIMENTS\n'+'='*80)

# Adaptive Top-K (F1>=0.75 captures XGB, LGBM, MP)
tk_mask=[cc_metrics[m]['f1']>=0.75 for m in model_keys_cc]
tk_names=[m for m,k in zip(model_keys_cc,tk_mask) if k]
tk_scores=raw_scores[:,tk_mask]
print(f'\nTop-K models (F1>=0.75): {tk_names}')

mid_mask=[cc_metrics[m]['f1']>=0.50 for m in model_keys_cc]
mid_names=[m for m,k in zip(model_keys_cc,mid_mask) if k]
mid_scores=raw_scores[:,mid_mask]

m1,p1,pd1,t1=lr_stack(raw_scores,y_cc,model_keys_cc,C=1.0,label='LR-all-C1')
m2,p2,pd2,t2=lr_stack(tk_scores,y_cc,tk_names,C=1.0,label='LR-topK-C1')
m3,p3,pd3,t3=lr_stack(tk_scores,y_cc,tk_names,C=0.5,label='LR-topK-C0.5')

all_f1=[cc_metrics[m]['f1'] for m in model_keys_cc]
m4,w4,pd4,t4=wavg(raw_scores,y_cc,model_keys_cc,all_f1,'WAvg-all')
tk_f1=[cc_metrics[m]['f1'] for m in tk_names]
m5,w5,pd5,t5=wavg(tk_scores,y_cc,tk_names,tk_f1,'WAvg-topK')

# 2-model: XGB+LGBM
pair_names=[m for m in model_keys_cc if m in ('xgb_ae_hybrid','lgbm_classifier')]
if len(pair_names)==2:
    pair_mask=[m in pair_names for m in model_keys_cc]
    pair_scores=raw_scores[:,pair_mask]
    m8,p8,pd8,t8=lr_stack(pair_scores,y_cc,pair_names,C=1.0,label='LR-XGB+LGBM')
else:
    m8={'f1':0}; p8=p1; pd8=pd1; t8=t1

all_m=[('LR_all',m1,p1,pd1,t1),('LR_topK',m2,p2,pd2,t2),('LR_topK_C05',m3,p3,pd3,t3),
       ('WAvg_all',m4,w4,pd4,t4),('WAvg_topK',m5,w5,pd5,t5),('LR_XGB_LGBM',m8,p8,pd8,t8)]
best=max(all_m,key=lambda x:x[1]['f1'])
print(f'\n>>> Best CC fusion: {best[0]} F1={best[1]["f1"]:.4f}')
best_cc_name,best_cc_m,best_cc_probs,best_cc_preds,best_cc_thr=best


CC FUSION EXPERIMENTS

Top-K models (F1>=0.75): ['lgbm_classifier', 'matrix_profile', 'xgb_ae_hybrid']
  LR-all-C1: P=0.8571 R=0.7959 F1=0.8254 ROC=0.9587
  LR-topK-C1: P=0.8830 R=0.8469 F1=0.8646 ROC=0.9428
  LR-topK-C0.5: P=0.9011 R=0.8367 F1=0.8677 ROC=0.9422
  WAvg-all: P=0.9000 R=0.8265 F1=0.8617 ROC=0.9685 W={'conv_ae': np.float64(0.165), 'ecod': np.float64(0.055), 'isolation_forest': np.float64(0.087), 'lgbm_classifier': np.float64(0.242), 'matrix_profile': np.float64(0.209), 'xgb_ae_hybrid': np.float64(0.241)}
  WAvg-topK: P=0.9310 R=0.8265 F1=0.8757 ROC=0.9707 W={'lgbm_classifier': np.float64(0.35), 'matrix_profile': np.float64(0.302), 'xgb_ae_hybrid': np.float64(0.347)}
  LR-XGB+LGBM: P=0.8830 R=0.8469 F1=0.8646 ROC=0.9428

>>> Best CC fusion: WAvg_topK F1=0.8757


In [10]:
# Cell 7 — NAB Fusion (per-model weighting + explainability)
def keep_runs(y,min_len=3):
    y=np.asarray(y,dtype=np.int8).copy(); i,n=0,len(y)
    while i<n:
        if y[i]==1:
            j=i
            while j<n and y[j]==1: j+=1
            if (j-i)<min_len: y[i:j]=0
            i=j
        else: i+=1
    return y
def point_adjust(yt,yp):
    yt=np.asarray(yt,dtype=np.int8); yp=np.asarray(yp,dtype=np.int8).copy()
    i,n=0,len(yt)
    while i<n:
        if yt[i]==1:
            j=i
            while j<n and yt[j]==1: j+=1
            if yp[i:j].any(): yp[i:j]=1
            i=j
        else: i+=1
    return yp

nab_mk=sorted(nab_data.keys())
print(f'NAB models for fusion: {nab_mk}')

if not nab_data:
    print('No NAB data.'); nab_strict={}; nab_pa={}; best_nab_strat='none'
else:
    # Per-model NAB F1 for weighting
    model_nab_f1={}
    for m in nab_mk:
        mY,mP=[],[]
        for eid in nab_data[m]:
            ent=nab_data[m][eid]; mY.append(ent['y_true'])
            if ent.get('preds') is not None: mP.append(ent['preds'])
        if mP:
            my,mp=np.concatenate(mY[:len(mP)]),np.concatenate(mP)
            model_nab_f1[m]=f1_score(my,mp,zero_division=0)
        else:
            model_nab_f1[m]=0.1
    print(f'NAB per-model F1: { {k:round(v,4) for k,v in model_nab_f1.items()} }')

    all_eids=set()
    for m in nab_mk: all_eids.update(nab_data[m].keys())
    all_eids=sorted(all_eids)
    print(f'Total unique entities: {len(all_eids)}')
    coverage={}
    for eid in all_eids:
        coverage[eid]=[m for m in nab_mk if eid in nab_data[m]]

    def nab_fuse(strategy='avg'):
        S_all,Y_all,P_all,PA_all=[],[],[],[]
        entity_explanations=[]
        for eid in sorted(all_eids):
            models=coverage[eid]
            if not models: continue
            ref_m=models[0]; yt=nab_data[ref_m][eid]['y_true']; n=len(yt)
            sc,ws=[],[]
            for m in models:
                s=nab_data[m][eid]['scores']
                if len(s)!=n: continue
                smin,smax=s.min(),s.max()
                sn=(s-smin)/(smax-smin+1e-12)
                sc.append(sn)
                ws.append(model_nab_f1.get(m,0.1)**2)
            if not sc: continue
            if strategy=='avg':
                avg=np.mean(sc,axis=0).astype(np.float32)
            elif strategy=='wavg':
                wn=np.array(ws); wn=wn/wn.sum()
                avg=np.average(sc,axis=0,weights=wn).astype(np.float32)
            elif strategy=='max':
                avg=np.max(sc,axis=0).astype(np.float32)
            else:
                avg=np.mean(sc,axis=0).astype(np.float32)
            if yt.sum()>0:
                qs=np.linspace(0.5,0.999,200); tl=np.unique(np.quantile(avg,qs))
                bt,bf=float(tl[-1]),-1.0
                for t in tl:
                    p=keep_runs((avg>=t).astype(np.int8),3)
                    f=f1_score(yt,p,zero_division=0)
                    if f>bf: bf,bt=float(f),float(t)
                thr=bt
            else: thr=float(np.percentile(avg,99.5))
            pf=keep_runs((avg>=thr).astype(np.int8),3); pa=point_adjust(yt,pf)
            S_all.append(avg); Y_all.append(yt); P_all.append(pf); PA_all.append(pa)
            # EXPLAINABILITY: log per-entity model contributions
            entity_explanations.append({
                'entity': eid,
                'models_used': models,
                'n_models': len(models),
                'true_anomalies': int(yt.sum()),
                'pred_anomalies': int(pf.sum()),
                'threshold': float(thr),
            })
        if not S_all: return {},{},[]
        S,Y,P,PA=np.concatenate(S_all),np.concatenate(Y_all),np.concatenate(P_all),np.concatenate(PA_all)
        strict={'strict_precision':precision_score(Y,P,zero_division=0),'strict_recall':recall_score(Y,P,zero_division=0),
                'strict_f1':f1_score(Y,P,zero_division=0),'strict_rocauc':roc_auc_score(Y,S) if len(np.unique(Y))==2 else float('nan')}
        pa_m={'pa_precision':precision_score(Y,PA,zero_division=0),'pa_recall':recall_score(Y,PA,zero_division=0),'pa_f1':f1_score(Y,PA,zero_division=0)}
        return strict,pa_m,entity_explanations

    print('\n'+'='*80)
    print('NAB ENSEMBLE — FUSION EXPERIMENTS')
    print('='*80)
    strats={}
    for strat in ['avg','wavg','max']:
        s,p,expl=nab_fuse(strat)
        if s:
            strats[strat]=(s,p,expl)
            print(f'  {strat:6s}: strict_F1={s["strict_f1"]:.4f}  PA_F1={p["pa_f1"]:.4f}  ROC={s.get("strict_rocauc",0):.4f}')
    best_nab_strat=max(strats,key=lambda k:strats[k][0]['strict_f1'])
    nab_strict,nab_pa,nab_explanations=strats[best_nab_strat]
    print(f'\n>>> Best NAB: {best_nab_strat} strict_F1={nab_strict["strict_f1"]:.4f}')

    # EXPLAINABILITY SUMMARY
    print('\n=== NAB ENSEMBLE EXPLAINABILITY ===')
    print(f'Fusion strategy: {best_nab_strat}')
    if best_nab_strat == 'wavg':
        wn = np.array([model_nab_f1.get(m,0.1)**2 for m in nab_mk])
        wn = wn / wn.sum()
        print(f'Model weights: {dict(zip(nab_mk, wn.round(4)))}')
    print(f'Entities fused: {len(nab_explanations)}')
    # Show top 5 best and worst performing entities
    for expl in nab_explanations:
        expl['hit_rate'] = expl['pred_anomalies'] / max(expl['true_anomalies'], 1)
    sorted_expl = sorted(nab_explanations, key=lambda x: x['hit_rate'], reverse=True)
    print('\nTop 5 best-detected entities:')
    for e in sorted_expl[:5]:
        print(f"  {e['entity']:40s} models={e['n_models']} true={e['true_anomalies']} pred={e['pred_anomalies']}")
    print('Bottom 5 worst-detected entities:')
    for e in sorted_expl[-5:]:
        print(f"  {e['entity']:40s} models={e['n_models']} true={e['true_anomalies']} pred={e['pred_anomalies']}")


NAB models for fusion: ['conv_ae', 'ecod', 'isolation_forest', 'matrix_profile']
NAB per-model F1: {'conv_ae': 0.3404, 'ecod': 0.2162, 'isolation_forest': 0.1937, 'matrix_profile': 0.1366}
Total unique entities: 58

NAB ENSEMBLE — FUSION EXPERIMENTS
  avg   : strict_F1=0.4536  PA_F1=0.7019  ROC=0.6809
  wavg  : strict_F1=0.4716  PA_F1=0.7018  ROC=0.6906
  max   : strict_F1=0.3813  PA_F1=0.6174  ROC=0.5728

>>> Best NAB: wavg strict_F1=0.4716

=== NAB ENSEMBLE EXPLAINABILITY ===
Fusion strategy: wavg
Model weights: {'conv_ae': np.float64(0.5296), 'ecod': np.float64(0.2137), 'isolation_forest': np.float64(0.1714), 'matrix_profile': np.float64(0.0853)}
Entities fused: 58

Top 5 best-detected entities:
  art_flatline.csv                         models=4 true=0 pred=1209
  ec2_network_in_5abac7.csv                models=4 true=17 pred=694
  Twitter_volume_AAPL.csv                  models=4 true=0 pred=24
  Twitter_volume_CRM.csv                   models=4 true=0 pred=24
  Twitter_volume_UPS

In [11]:
# Cell 8 — Individual NAB metrics for comparison
if nab_data:
    print('\nINDIVIDUAL NAB MODEL METRICS:')
    for m in nab_mk:
        mY,mP,mPA=[],[],[]
        for eid in nab_data[m]:
            ent=nab_data[m][eid]; mY.append(ent['y_true'])
            if ent.get('preds') is not None:
                mP.append(ent['preds']); mPA.append(point_adjust(ent['y_true'],ent['preds']))
        if mP:
            my,mp,mpa=np.concatenate(mY[:len(mP)]),np.concatenate(mP),np.concatenate(mPA)
            sf1=f1_score(my,mp,zero_division=0); pf1=f1_score(my,mpa,zero_division=0)
            print(f'  {m:20s} strict_F1={sf1:.4f}  PA_F1={pf1:.4f}  entities={len(nab_data[m])}')
        else:
            print(f'  {m:20s} (no preds available)')



INDIVIDUAL NAB MODEL METRICS:
  conv_ae              strict_F1=0.3404  PA_F1=0.4991  entities=58
  ecod                 strict_F1=0.2162  PA_F1=0.5150  entities=58
  isolation_forest     strict_F1=0.1937  PA_F1=0.4576  entities=58
  matrix_profile       strict_F1=0.1366  PA_F1=0.2979  entities=58


In [12]:
# Cell 9 — Final Comparison + Automated Weighting Report
print('='*80)
print('FINAL COMPARISON — AUTOMATED ENSEMBLE ANALYSIS')
print('='*80)

# --- CC Results ---
rows=[]
for mk,m in cc_metrics.items(): rows.append({'method':mk,'cc_f1':m['f1'],'cc_rocauc':m['rocauc'],'cc_prauc':m['prauc']})
rows.append({'method':f'ENSEMBLE_{best_cc_name}','cc_f1':best_cc_m['f1'],'cc_rocauc':best_cc_m['rocauc'],'cc_prauc':best_cc_m['prauc']})
print('\nCredit Card:'); display(pd.DataFrame(rows).sort_values('cc_f1',ascending=False))
bi_f1=max(m['f1'] for m in cc_metrics.values()); bi_n=max(cc_metrics,key=lambda k:cc_metrics[k]['f1'])
print(f'Best individual: {bi_n} F1={bi_f1:.4f}')
print(f'Best ensemble  : {best_cc_name} F1={best_cc_m["f1"]:.4f}')
if best_cc_m['f1']>bi_f1: print(f'>>> ENSEMBLE WINS by {best_cc_m["f1"]-bi_f1:.4f}')
else: print(f'>>> Gap: {bi_f1-best_cc_m["f1"]:.4f}')

# --- NAB Results ---
if nab_strict:
    print('\nNAB:')
    nr=[]
    for m in nab_mk:
        mY,mP,mPA=[],[],[]
        for eid in nab_data[m]:
            ent=nab_data[m][eid]; mY.append(ent['y_true'])
            if ent.get('preds') is not None: mP.append(ent['preds']); mPA.append(point_adjust(ent['y_true'],ent['preds']))
        if mP:
            my,mp,mpa=np.concatenate(mY[:len(mP)]),np.concatenate(mP),np.concatenate(mPA)
            nr.append({'method':m,'strict_f1':f1_score(my,mp,zero_division=0),'pa_f1':f1_score(my,mpa,zero_division=0),'entities':len(nab_data[m])})
    nr.append({'method':f'ENSEMBLE_{best_nab_strat}','strict_f1':nab_strict['strict_f1'],'pa_f1':nab_pa['pa_f1'],'entities':len(all_eids)})
    display(pd.DataFrame(nr).sort_values('strict_f1',ascending=False))

# --- AUTOMATED WEIGHT REPORT ---
print('\n' + '='*80)
print('AUTOMATED WEIGHTING REPORT')
print('='*80)
print('\nCC Model Weights (F1^2 normalized):')
cc_w = {m: cc_metrics[m]['f1']**2 for m in model_keys_cc}
cc_w_sum = sum(cc_w.values())
for m in sorted(cc_w, key=cc_w.get, reverse=True):
    pct = 100*cc_w[m]/cc_w_sum
    bar = '█' * int(pct/2)
    print(f'  {m:20s}: weight={cc_w[m]/cc_w_sum:.4f} ({pct:.1f}%) {bar}')
print(f'\n  Top-3 models carry {sum(sorted(cc_w.values(),reverse=True)[:3])/cc_w_sum*100:.1f}% of total weight')

if nab_strict and best_nab_strat == 'wavg':
    print('\nNAB Model Weights (F1^2 normalized):')
    nab_w = {m: model_nab_f1.get(m,0.1)**2 for m in nab_mk}
    nab_w_sum = sum(nab_w.values())
    for m in sorted(nab_w, key=nab_w.get, reverse=True):
        pct = 100*nab_w[m]/nab_w_sum
        bar = '█' * int(pct/2)
        print(f'  {m:20s}: weight={nab_w[m]/nab_w_sum:.4f} ({pct:.1f}%) {bar}')

# --- DECISION EXPLAINABILITY ---
print('\n' + '='*80)
print('ENSEMBLE DECISION EXPLAINABILITY — CREDIT CARD')
print('='*80)
# Show which models agree/disagree on specific predictions
all_preds = {}
for mk in model_keys_cc:
    d = cc_data[mk]
    p = d['preds'] if d['preds'] is not None else (d['scores'] >= (d['threshold'] or np.percentile(d['scores'],99))).astype(np.int8)
    all_preds[mk] = p

# Agreement analysis
n_samples = len(y_cc)
agreement = np.zeros(n_samples, dtype=int)
for mk in model_keys_cc:
    agreement += all_preds[mk]

print(f'Full agreement (all {len(model_keys_cc)} models say fraud): {(agreement == len(model_keys_cc)).sum()} samples')
print(f'Majority agree (>50% say fraud): {(agreement > len(model_keys_cc)/2).sum()} samples')
print(f'No model flags: {(agreement == 0).sum()} samples')
print(f'Exactly 1 model flags: {(agreement == 1).sum()} samples')

# For actual frauds, how many models catch them?
fraud_idx = np.where(y_cc == 1)[0]
print(f'\nFor {len(fraud_idx)} actual frauds:')
for n_agree in range(len(model_keys_cc)+1):
    caught = ((agreement[fraud_idx] >= n_agree) & (agreement[fraud_idx] <= n_agree+0.5)).sum()
    if caught > 0:
        print(f'  Caught by exactly {n_agree}/{len(model_keys_cc)} models: {caught} frauds')


FINAL COMPARISON — AUTOMATED ENSEMBLE ANALYSIS

Credit Card:


,method,cc_f1,cc_rocauc,cc_prauc
6,ENSEMBLE_WAvg_topK,0.875676,0.970746,0.852944
3,lgbm_classifier,0.869565,0.937231,0.848038
5,xgb_ae_hybrid,0.866310,0.980868,0.873092
4,matrix_profile,0.808290,0.964591,0.695749
0,conv_ae,0.717949,0.957455,0.687445
2,isolation_forest,0.520179,0.959610,0.506539
1,ecod,0.416000,0.961591,0.353613


Best individual: lgbm_classifier F1=0.8696
Best ensemble  : WAvg_topK F1=0.8757
>>> ENSEMBLE WINS by 0.0061

NAB:


,method,strict_f1,pa_f1,entities
4,ENSEMBLE_wavg,0.471555,0.701769,58
0,conv_ae,0.340394,0.499148,58
1,ecod,0.216239,0.514990,58
2,isolation_forest,0.193672,0.457561,58
3,matrix_profile,0.136584,0.297918,58



AUTOMATED WEIGHTING REPORT

CC Model Weights (F1^2 normalized):
  lgbm_classifier     : weight=0.2424 (24.2%) ████████████
  xgb_ae_hybrid       : weight=0.2406 (24.1%) ████████████
  matrix_profile      : weight=0.2095 (20.9%) ██████████
  conv_ae             : weight=0.1653 (16.5%) ████████
  isolation_forest    : weight=0.0868 (8.7%) ████
  ecod                : weight=0.0555 (5.5%) ██

  Top-3 models carry 69.3% of total weight

NAB Model Weights (F1^2 normalized):
  conv_ae             : weight=0.5296 (53.0%) ██████████████████████████
  ecod                : weight=0.2137 (21.4%) ██████████
  isolation_forest    : weight=0.1714 (17.1%) ████████
  matrix_profile      : weight=0.0853 (8.5%) ████

ENSEMBLE DECISION EXPLAINABILITY — CREDIT CARD
Full agreement (all 6 models say fraud): 48 samples
Majority agree (>50% say fraud): 83 samples
No model flags: 56759 samples
Exactly 1 model flags: 50 samples

For 98 actual frauds:
  Caught by exactly 0/6 models: 13 frauds
  Caught by exact

In [13]:
# Cell 10 — Export with explainability artifacts
cc_ens={'dataset':'creditcard','model':f'coordinator_v3_{best_cc_name}','base_models':model_keys_cc,
    'fusion_strategy': best_cc_name,
    'entities':{'creditcard':{'entityid':'creditcard','scoresfull':best_cc_probs.astype(np.float32),
        'yfull':y_cc,'predfull':best_cc_preds,'originalrowid':ref_rid,'threshold':best_cc_thr}},
    'metrics':best_cc_m,'individual_metrics':cc_metrics,
    'model_weights': {m: float(cc_metrics[m]['f1']**2) for m in model_keys_cc}}
joblib.dump(cc_ens,os.path.join(OUTPUTDIR,'coordinator_v3_creditcard.joblib'))
print('Saved CC ensemble')

if nab_strict:
    nab_export = {'dataset':'nab','model':f'coordinator_v3_{best_nab_strat}','base_models':nab_mk,
        'fusion_strategy':best_nab_strat,'metrics':{**nab_strict,**nab_pa},
        'model_weights': {m: float(model_nab_f1.get(m,0.1)**2) for m in nab_mk}}
    if 'nab_explanations' in dir():
        nab_export['entity_explanations'] = nab_explanations
    joblib.dump(nab_export, os.path.join(OUTPUTDIR,'coordinator_v3_nab.joblib'))
    print('Saved NAB ensemble (with explanations)')

rows=[{'dataset':'creditcard','method':best_cc_name,**best_cc_m,'n_models':len(model_keys_cc)}]
if nab_strict: rows.append({'dataset':'nab','method':best_nab_strat,**nab_strict,**nab_pa,'n_models':len(nab_mk)})
s=pd.DataFrame(rows); s.to_csv(os.path.join(OUTPUTDIR,'coordinator_v3_summary.csv'),index=False)
print('Saved summary'); display(s)

with open(os.path.join(OUTPUTDIR,'coordinator_v3_manifest.json'),'w') as f:
    json.dump({'runid':RUNID,'version':'v3.3',
        'best_cc':best_cc_name,'best_nab':best_nab_strat if nab_strict else None,
        'cc_models':model_keys_cc,'nab_models':nab_mk,
        'cc_weights': {m: round(cc_metrics[m]['f1']**2, 6) for m in model_keys_cc},
        'nab_weights': {m: round(model_nab_f1.get(m,0.1)**2, 6) for m in nab_mk} if nab_strict else {},
        'explainability': True,
        'outputdir':OUTPUTDIR},f,indent=2)

print('\nCoordinator v3.3 complete (with explainability + automated weighting).')
print('Outputs:', OUTPUTDIR)


Saved CC ensemble
Saved NAB ensemble (with explanations)
Saved summary


,dataset,method,precision,recall,f1,rocauc,prauc,n_models,strict_precision,strict_recall,strict_f1,strict_rocauc,pa_precision,pa_recall,pa_f1
0,creditcard,WAvg_topK,0.931034,0.826531,0.875676,0.970746,0.852944,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,nab,wavg,NaN,NaN,NaN,NaN,NaN,4,0.405458,0.563399,0.471555,0.690647,0.544394,0.987134,0.701769



Coordinator v3.3 complete (with explainability + automated weighting).
Outputs: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/coordinator_v3/predictions
